# Multi-Agent LinkedIn Post Generator (Qwen2.5-3B-Instruct)

Pipeline: **Prompt Refiner → Generator → Validator → Corrector (loop)**

Single 3B model, four different system prompts ("agents"). Plain Python state machine, no framework overhead — built to run end-to-end on a Colab T4/L4 GPU.

## 1. Setup

In [1]:
!pip install -q -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 39.8 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import json, re

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print("CUDA available:", torch.cuda.is_available())

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print("Model loaded.")

CUDA available: True


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


## 2. Core model call wrapper

In [16]:
def call_model(system_prompt: str, user_prompt: str, max_new_tokens: int = 500, temperature: float = 0.7, min_new_tokens: int = 0) -> str:
    """Single chat-style call to the model. Returns raw text output."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,   # forces the model to keep generating (fixes short drafts)
            temperature=temperature,
            do_sample=temperature > 0,
            top_p=0.9,
            repetition_penalty=1.15,          # discourages filler repetition when forced to run longer
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 3. Agent prompts

In [17]:
REFINER_SYSTEM = """You are a Prompt Engineering Agent.
Your ONLY job is to turn raw, unstructured user input into a clean, well-structured generation prompt for another LLM that will write a LinkedIn post.

Apply these prompt-engineering rules:
- Assign the writer LLM a clear role (e.g. "You are a professional LinkedIn content writer").
- State the task explicitly.
- List hard constraints: word count range, tone, structure (hook / body / CTA), hashtag policy, emoji policy.
- Include the key points/facts the user gave, verbatim, so nothing is lost.
- Forbid generic cliches (e.g. "In today's fast-paced world", "I'm excited to announce").
- End with a clear output format instruction: "Output ONLY the post text, nothing else."

Output ONLY the refined prompt text. No preamble, no explanation, no markdown fences."""

GENERATOR_SYSTEM = """You are a professional LinkedIn content writer.
Follow the instructions given to you exactly and precisely, including the word count range.
Output ONLY the final LinkedIn post text. No preamble, no explanation, no markdown fences, no quotation marks around the post."""

VALIDATOR_SYSTEM = """You are a strict Quality Validator Agent for LinkedIn posts.
Evaluate the DRAFT against this rubric:
1. Hook: does the first line grab attention (no generic opener)?
2. Length: is it within the requested word range?
3. Tone: does it match the requested tone?
4. Cliches: free of generic filler phrases ("In today's fast-paced world", "I'm excited to announce", excessive buzzwords)?
5. CTA: is there a clear call-to-action or closing thought?
6. Emoji/hashtag policy: respected as requested?
7. Factual consistency: does it match the key points provided, with no invented facts?

Respond with ONLY a single valid JSON object, nothing else, in exactly this schema:
{"pass": true or false, "issues": ["short issue 1", "short issue 2"]}

If everything passes, return {"pass": true, "issues": []}.
Do not include any text outside the JSON object."""

CORRECTOR_SYSTEM = """You are a Correction Agent for LinkedIn posts.
You will be given the ORIGINAL instructions, the current DRAFT, and a list of ISSUES found by a validator.
Make the MINIMAL edits necessary to fix exactly those issues. Do not rewrite parts of the post that were not flagged.
Preserve the original structure, facts, and tone unless an issue explicitly requires changing them.

Output ONLY the corrected post text. No preamble, no explanation, no markdown fences."""

EXPANDER_SYSTEM = """You are an Expansion Agent for LinkedIn posts.
You will be given ORIGINAL instructions, a DRAFT that is too SHORT, and a target word range.

Your job: rewrite the post to reach the target word count by adding genuine substance, not filler:
- Expand the story/example with more concrete detail (numbers, moments, specific struggles or decisions).
- Add a second supporting point or a short anecdote if the topic allows it.
- Deepen the reflection/insight paragraph.
- Do NOT pad with repeated sentences, generic filler, or restating the same idea in different words.
- Keep the same tone, facts, and CTA as the original draft.

You MUST produce a post within the target word range. Falling short is a failure.
Output ONLY the expanded post text. No preamble, no explanation, no markdown fences."""

## 4. JSON parsing helper (robust to small-model messiness)

In [18]:
def parse_validator_output(raw: str) -> dict:
    """Try to extract the JSON verdict from the validator's raw output."""
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass

    # Fallback: assume pass if we truly can't parse, but flag it
    print("[WARN] Could not parse validator JSON, raw output was:\n", raw)
    return {"pass": True, "issues": ["validator_output_unparseable"]}

In [22]:
def parse_word_target(word_count_str: str) -> tuple[int, int]:
    """Extract (min_words, max_words) from a string like '250-500' or '150'."""
    nums = [int(n) for n in re.findall(r"\d+", word_count_str)]
    if len(nums) >= 2:
        return min(nums), max(nums)
    elif len(nums) == 1:
        return nums[0], nums[0]
    return 120, 180  # fallback default


def words_in(text: str) -> int:
    return len(text.split())


def check_word_count(draft: str, word_count_str: str) -> str | None:
    """Returns an issue string if out of range, else None. Deterministic check — don't trust the small validator model to count."""
    lo, hi = parse_word_target(word_count_str)
    n = words_in(draft)
    if n < lo:
        return f"Draft is {n} words, below the required {lo}-{hi} word range."
    if n > hi:
        return f"Draft is {n} words, above the required {lo}-{hi} word range."
    return None

import unicodedata

EMOJI_PATTERN = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002600-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "\u2700-\u27bf\u2600-\u26ff\u2b00-\u2bff"
    "]+", flags=re.UNICODE
)

META_PATTERNS = [
    r"note:\s*links? have been placeholders",
    r"replace (this|them) (with|accordingly)",
    r"\[link to the library\]",
    r"\[github repository",
    r"\[contact information",
    r"\[testimonial section",
    r"if you need more assistance",
    r"\[replace this with",
    r"additional note:",
    r"clickable ones",
]
META_RE = re.compile("|".join(META_PATTERNS), flags=re.IGNORECASE)


def check_emojis(draft: str, emoji_policy: str) -> str | None:
    count = len(EMOJI_PATTERN.findall(draft))
    policy = emoji_policy.lower()
    if "none" in policy and count > 0:
        return f"Draft contains {count} emoji(s) but the policy is 'none' — remove all emojis."
    m = re.search(r"max\s*(\d+)", policy)
    if m and count > int(m.group(1)):
        return f"Draft contains {count} emoji(s), exceeding the max of {m.group(1)}."
    return None


def check_meta_commentary(draft: str) -> str | None:
    if META_RE.search(draft):
        return "Draft contains meta-commentary or placeholder notes addressed to the user (e.g. 'replace this link', 'Note:') — remove them; the post must be the final publish-ready text only."
    return None


def clean_draft(draft: str) -> str:
    """Belt-and-suspenders cleanup applied to the final draft regardless of validator verdict."""
    lines = draft.split("\n")
    cleaned = [ln for ln in lines if not META_RE.search(ln)]
    text = "\n".join(cleaned)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text

## 5. Orchestrator (plain Python state machine)

In [23]:
MAX_ITERATIONS = 3

def generate_linkedin_post(user_input: dict, verbose: bool = True) -> dict:
    """
    user_input example:
    {
        "topic": "Launching my new open-source ML library",
        "audience": "AI engineers and recruiters",
        "tone": "confident but humble",
        "key_points": ["3 months of work", "cuts inference latency by 40%", "MIT licensed"],
        "cta": "Ask them to try it and give feedback",
        "word_count": "120-180",
        "hashtags": "3-5 relevant hashtags",
        "emojis": "minimal, max 2"
    }
    """
    state = {
        "raw_input": user_input,
        "refined_prompt": None,
        "draft": None,
        "validation_history": [],
        "iteration": 0,
    }

    lo, hi = parse_word_target(user_input.get("word_count", "120-180"))
    gen_max_tokens = int(hi * 1.6) + 50   # headroom so the model isn't cut off mid-sentence
    gen_min_tokens = int(lo * 1.3)        # forces the model to actually reach the lower bound

    # --- Step 1: Prompt Refiner ---
    if verbose: print("="*60, "\n[1] Prompt Refiner Agent running...\n")
    refiner_input = json.dumps(user_input, indent=2)
    refined_prompt = call_model(REFINER_SYSTEM, refiner_input, max_new_tokens=400, temperature=0.3)
    state["refined_prompt"] = refined_prompt
    if verbose: print(refined_prompt, "\n")

    # --- Step 2: Generator ---
    if verbose: print("="*60, "\n[2] Generator Agent running...\n")
    draft = call_model(GENERATOR_SYSTEM, refined_prompt, max_new_tokens=gen_max_tokens, min_new_tokens=gen_min_tokens, temperature=0.8)
    state["draft"] = draft
    if verbose: print(draft, f"\n[{words_in(draft)} words]\n")

    # --- Step 3/4: Validator <-> Corrector/Expander loop ---
    for i in range(MAX_ITERATIONS):
        state["iteration"] = i + 1
        if verbose: print("="*60, f"\n[3] Validator Agent running (round {i+1})...\n")

        validator_input = (
            f"ORIGINAL INSTRUCTIONS:\n{refined_prompt}\n\n"
            f"DRAFT:\n{draft}"
        )
        raw_validation = call_model(VALIDATOR_SYSTEM, validator_input, max_new_tokens=250, temperature=0.0)
        verdict = parse_validator_output(raw_validation)

        # Hard, code-based word-count check -- don't trust the small model to count
        for issue_fn, arg in [
            (lambda d: check_word_count(d, user_input.get("word_count", "120-180")), draft),
            (lambda d: check_emojis(d, user_input.get("emojis", "minimal, max 2")), draft),
            (lambda d: check_meta_commentary(d), draft),
        ]:
            issue = issue_fn(arg)
            if issue:
                verdict["pass"] = False
                verdict.setdefault("issues", []).append(issue)

        state["validation_history"].append(verdict)
        if verbose: print(json.dumps(verdict, indent=2), f"\n[{words_in(draft)} words]\n")

        if verdict.get("pass", False):
            if verbose: print("Validation PASSED. Done.\n")
            break

        if i == MAX_ITERATIONS - 1:
            if verbose: print("Max iterations reached. Returning best draft with remaining issues flagged.\n")
            break

        issues = verdict.get("issues", [])
        needs_expansion = any(
            "word" in issue.lower() and ("below" in issue.lower() or "short" in issue.lower())
            for issue in issues
        )

        if needs_expansion:
            if verbose: print("="*60, f"\n[4] Expander Agent running (round {i+1})...\n")
            expander_input = (
                f"ORIGINAL INSTRUCTIONS:\n{refined_prompt}\n\n"
                f"CURRENT DRAFT ({words_in(draft)} words):\n{draft}\n\n"
                f"TARGET WORD RANGE: {lo}-{hi} words. Expand it to reach this range."
            )
            draft = call_model(
                EXPANDER_SYSTEM, expander_input,
                max_new_tokens=gen_max_tokens, min_new_tokens=gen_min_tokens, temperature=0.7,
            )
        else:
            if verbose: print("="*60, f"\n[4] Corrector Agent running (round {i+1})...\n")
            corrector_input = (
                f"ORIGINAL INSTRUCTIONS:\n{refined_prompt}\n\n"
                f"DRAFT:\n{draft}\n\n"
                f"ISSUES TO FIX:\n{json.dumps(issues, indent=2)}"
            )
            draft = call_model(CORRECTOR_SYSTEM, corrector_input, max_new_tokens=gen_max_tokens, temperature=0.5)

        state["draft"] = draft
        if verbose: print(draft, f"\n[{words_in(draft)} words]\n")

    state["final_post"] = draft
    return state

## 6. Run it

In [27]:
user_input = {
    "topic": "Launching my new open-source ML library",
    "audience": "AI engineers and recruiters",
    "tone": "confident but humble",
    "key_points": [
        "3 months of work",
        "cuts inference latency by 40%",
        "MIT licensed"
    ],
    "cta": "Ask them to try it and give feedback",
    "word_count": "150-300",
    "hashtags": "3-5 relevant hashtags",
    "emojis": "none"
}

result = generate_linkedin_post(user_input, verbose=True)

[1] Prompt Refiner Agent running...

You are a professional LinkedIn content writer tasked with crafting an engaging LinkedIn post about launching a new open-source machine learning library. The topic focuses on AI engineers and recruiters interested in cutting-edge technology. Your tone should be confident yet humble, highlighting three main achievements: 3 months of dedicated development time, significant performance improvement (a 40% reduction in inference latency), and its license being MIT. 

The post must adhere to the following guidelines:
- Keep your message between 150 to 300 words inclusive.
- Use a tone that reflects confidence while maintaining humility.
- Structure your post as follows: Hook (introduce the project briefly), Body (details including key achievements), Call-to-action (CTA) encouraging readers to test and provide feedback.
- Limit your use of hashtags to 3-5 relevant ones.
- Avoid using emojis entirely.

Please include the provided key points verbatim:

"3 mo

In [28]:
print("FINAL POST:\n")
print(result["final_post"])

FINAL POST:

Launching our latest endeavor, we’re thrilled to introduce [Project Name], an innovative open-source machine learning library tailored for AI engineers eager to leverage cutting-edge technologies. Our journey began when we set out to develop a solution that could streamline complex computations, particularly focusing on reducing inference latency—critical in real-time applications where every millisecond counts. After just 3 months of relentless effort, we achieved remarkable milestones: significantly cutting inference latency by an astounding 40%.

Underpinning these improvements is our commitment to transparency and community-driven innovation. We chose the MIT license, which ensures broad accessibility and adaptability without imposing any restrictions on derivative works or commercial usage. Unlike restrictive licenses, MIT fosters collaboration among developers worldwide.

Join us in revolutionizing AI practices. Test [Project Name] firsthand, offer constructive feedb

## 7. Simple interactive CLI cell
Run this cell and fill in the fields to generate a post for your own input.

In [29]:
def build_input_interactively():
    print("Enter details for your LinkedIn post (press Enter to skip a field):\n")
    topic = input("Topic: ")
    audience = input("Audience: ")
    tone = input("Tone (e.g. confident, casual, formal): ")
    key_points_raw = input("Key points (comma-separated): ")
    cta = input("Call to action: ")
    word_count = input("Word count range (default 120-180): ") or "120-180"
    hashtags = input("Hashtag policy (default '3-5 relevant hashtags'): ") or "3-5 relevant hashtags"
    emojis = input("Emoji policy (default 'minimal, max 2'): ") or "minimal, max 2"

    return {
        "topic": topic,
        "audience": audience,
        "tone": tone,
        "key_points": [p.strip() for p in key_points_raw.split(",") if p.strip()],
        "cta": cta,
        "word_count": word_count,
        "hashtags": hashtags,
        "emojis": emojis,
    }

# my_input = build_input_interactively()
# my_result = generate_linkedin_post(my_input, verbose=True)
# print("\nFINAL POST:\n")
# print(my_result["final_post"])

## 8. Interactive UI (ipywidgets)

A proper form UI — fill in the fields, click **Generate**, watch each agent's output stream into its own log panel, and get the final post in a copyable box. Works directly in Colab, no extra install needed (ipywidgets ships with Colab).

In [32]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, HTML

# --- Inject custom CSS once ---
display(HTML("""
<style>
.lp-card {
    background: #ffffff;
    border: 1px solid #e2e5e9;
    border-radius: 14px;
    padding: 20px 24px;
    margin-bottom: 14px;
    box-shadow: 0 1px 3px rgba(0,0,0,0.06);
}
.lp-title {
    font-size: 22px;
    font-weight: 700;
    background: linear-gradient(90deg, #0a66c2, #378fe9);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    margin-bottom: 4px;
}
.lp-subtitle {
    color: #6b7280;
    font-size: 13px;
    margin-bottom: 16px;
}
.lp-section-label {
    font-size: 11px;
    font-weight: 700;
    letter-spacing: 0.06em;
    text-transform: uppercase;
    color: #0a66c2;
    margin: 14px 0 6px 0;
}
.lp-final-card {
    background: linear-gradient(135deg, #f0f7ff, #ffffff);
    border: 1px solid #cfe3fb;
    border-radius: 14px;
    padding: 22px 26px;
    box-shadow: 0 2px 10px rgba(10,102,194,0.08);
}
.lp-final-header {
    display: flex;
    align-items: center;
    gap: 8px;
    font-weight: 700;
    font-size: 15px;
    color: #0a66c2;
    margin-bottom: 10px;
}
.lp-badge {
    display: inline-block;
    background: #e7f1fd;
    color: #0a66c2;
    font-size: 11px;
    font-weight: 700;
    padding: 3px 10px;
    border-radius: 999px;
}
.lp-post-text {
    white-space: pre-wrap;
    font-family: -apple-system, Segoe UI, Roboto, sans-serif;
    font-size: 14.5px;
    line-height: 1.65;
    color: #1c1e21;
}
.widget-text input, .widget-textarea textarea, .widget-dropdown select {
    border-radius: 8px !important;
    border: 1px solid #d5d9dd !important;
}
.widget-text input:focus, .widget-textarea textarea:focus {
    border-color: #0a66c2 !important;
    box-shadow: 0 0 0 2px rgba(10,102,194,0.15) !important;
}
.lp-generate-btn > button {
    background: linear-gradient(90deg, #0a66c2, #2e8fef) !important;
    color: white !important;
    border: none !important;
    border-radius: 10px !important;
    font-weight: 600 !important;
    box-shadow: 0 2px 8px rgba(10,102,194,0.35) !important;
    transition: transform 0.1s ease;
}
.lp-generate-btn > button:hover {
    transform: translateY(-1px);
}
</style>
"""))

# --- Input widgets ---
w_topic = widgets.Text(placeholder="Launching my new open-source ML library", layout=widgets.Layout(width="100%"))
w_audience = widgets.Text(placeholder="AI engineers and recruiters", layout=widgets.Layout(width="100%"))
w_tone = widgets.Dropdown(
    options=["confident but humble", "casual", "formal", "inspirational", "witty", "reflective", "bold/contrarian"],
    layout=widgets.Layout(width="100%"),
)
w_key_points = widgets.Textarea(
    placeholder="One per line, e.g.\n3 months of work\ncuts inference latency by 40%\nMIT licensed",
    layout=widgets.Layout(width="100%", height="80px"),
)
w_cta = widgets.Text(placeholder="Ask them to try it and give feedback", layout=widgets.Layout(width="100%"))
w_word_count = widgets.Text(value="120-180", layout=widgets.Layout(width="100%"))
w_hashtags = widgets.Text(value="3-5 relevant hashtags", layout=widgets.Layout(width="100%"))
w_emojis = widgets.Text(value="minimal, max 2", layout=widgets.Layout(width="100%"))

def labeled(label, widget):
    return widgets.VBox([widgets.HTML(f"<div style='font-size:12.5px;font-weight:600;color:#374151;margin-bottom:3px'>{label}</div>"), widget])

w_generate_btn = widgets.Button(description="✨  Generate Post", layout=widgets.Layout(width="220px", height="42px"))
w_generate_btn.add_class("lp-generate-btn")
w_status = widgets.HTML(value="")

# --- Output areas ---
out_refiner = widgets.Output()
out_generator = widgets.Output()
out_validator = widgets.Output()
out_final = widgets.Output()

agent_accordion = widgets.Accordion(children=[out_refiner, out_generator, out_validator])
agent_accordion.set_title(0, "🧭  1) Prompt Refiner")
agent_accordion.set_title(1, "✍️  2) Generator")
agent_accordion.set_title(2, "🔍  3–4) Validator / Corrector / Expander rounds")
agent_accordion.selected_index = None

def on_generate_clicked(b):
    for o in (out_refiner, out_generator, out_validator, out_final):
        o.clear_output()
    w_status.value = "<span style='color:#0a66c2;font-weight:600'>⏳ Running pipeline...</span>"
    w_generate_btn.disabled = True

    user_input = {
        "topic": w_topic.value.strip(),
        "audience": w_audience.value.strip(),
        "tone": w_tone.value,
        "key_points": [p.strip() for p in w_key_points.value.split("\n") if p.strip()],
        "cta": w_cta.value.strip(),
        "word_count": w_word_count.value.strip() or "120-180",
        "hashtags": w_hashtags.value.strip() or "3-5 relevant hashtags",
        "emojis": w_emojis.value.strip() or "minimal, max 2",
    }

    if not user_input["topic"]:
        w_status.value = "<span style='color:#c0392b;font-weight:600'>⚠️ Please enter at least a topic.</span>"
        w_generate_btn.disabled = False
        return

    try:
        with out_refiner:
            print("Input:\n", json.dumps(user_input, indent=2))
        refined_prompt = call_model(REFINER_SYSTEM, json.dumps(user_input, indent=2), max_new_tokens=400, temperature=0.3)
        with out_refiner:
            print("\nRefined prompt:\n", refined_prompt)

        lo, hi = parse_word_target(user_input["word_count"])
        gen_max_tokens = int(hi * 1.6) + 50
        gen_min_tokens = int(lo * 1.3)

        draft = call_model(GENERATOR_SYSTEM, refined_prompt, max_new_tokens=gen_max_tokens, min_new_tokens=gen_min_tokens, temperature=0.8)
        with out_generator:
            print(f"Draft v1 [{words_in(draft)} words]:\n", draft)

        for i in range(MAX_ITERATIONS):
            w_status.value = f"<span style='color:#0a66c2;font-weight:600'>🔎 Validating (round {i+1}/{MAX_ITERATIONS})...</span>"
            validator_input = f"ORIGINAL INSTRUCTIONS:\n{refined_prompt}\n\nDRAFT:\n{draft}"
            raw_validation = call_model(VALIDATOR_SYSTEM, validator_input, max_new_tokens=250, temperature=0.0)
            verdict = parse_validator_output(raw_validation)

            for issue_fn, arg in [
                (lambda d: check_word_count(d, user_input.get("word_count", "120-180")), draft),
                (lambda d: check_emojis(d, user_input.get("emojis", "minimal, max 2")), draft),
                (lambda d: check_meta_commentary(d), draft),
            ]:
                issue = issue_fn(arg)
                if issue:
                    verdict["pass"] = False
                    verdict.setdefault("issues", []).append(issue)

            with out_validator:
                print(f"--- Round {i+1} verdict [{words_in(draft)} words] ---")
                print(json.dumps(verdict, indent=2))

            if verdict.get("pass", False):
                break
            if i == MAX_ITERATIONS - 1:
                break

            issues = verdict.get("issues", [])
            needs_expansion = any(
                "word" in issue.lower() and ("below" in issue.lower() or "short" in issue.lower())
                for issue in issues
            )

            if needs_expansion:
                w_status.value = f"<span style='color:#0a66c2;font-weight:600'>📈 Expanding draft (round {i+1}/{MAX_ITERATIONS})...</span>"
                expander_input = (
                    f"ORIGINAL INSTRUCTIONS:\n{refined_prompt}\n\n"
                    f"CURRENT DRAFT ({words_in(draft)} words):\n{draft}\n\n"
                    f"TARGET WORD RANGE: {lo}-{hi} words. Expand it to reach this range."
                )
                draft = call_model(
                    EXPANDER_SYSTEM, expander_input,
                    max_new_tokens=gen_max_tokens, min_new_tokens=gen_min_tokens, temperature=0.7,
                )
                with out_validator:
                    print(f"\nExpanded draft (round {i+1}) [{words_in(draft)} words]:\n", draft, "\n")
            else:
                w_status.value = f"<span style='color:#0a66c2;font-weight:600'>🛠️ Correcting (round {i+1}/{MAX_ITERATIONS})...</span>"
                corrector_input = (
                    f"ORIGINAL INSTRUCTIONS:\n{refined_prompt}\n\n"
                    f"DRAFT:\n{draft}\n\n"
                    f"ISSUES TO FIX:\n{json.dumps(issues, indent=2)}"
                )
                draft = call_model(CORRECTOR_SYSTEM, corrector_input, max_new_tokens=gen_max_tokens, temperature=0.5)
                with out_validator:
                    print(f"\nCorrected draft (round {i+1}) [{words_in(draft)} words]:\n", draft, "\n")

        draft = clean_draft(draft)

        with out_final:
            display(HTML(f"""
            <div class="lp-final-card">
                <div class="lp-final-header">✅ Final Post <span class="lp-badge">{words_in(draft)} words</span></div>
                <div class="lp-post-text">{draft}</div>
            </div>
            """))
        w_status.value = "<span style='color:#0a8f3c;font-weight:600'>✓ Done</span> — expand the sections below to see each agent's reasoning"
    except Exception as e:
        w_status.value = f"<span style='color:#c0392b;font-weight:600'>⚠️ Error: {e}</span>"
        raise
    finally:
        w_generate_btn.disabled = False

w_generate_btn.on_click(on_generate_clicked)

input_card = widgets.VBox([
    widgets.HTML("<div class='lp-title'>📝 LinkedIn Post Generator</div><div class='lp-subtitle'>Multi-agent pipeline — Refiner → Generator → Validator → Corrector/Expander</div>"),
    labeled("Topic", w_topic),
    labeled("Audience", w_audience),
    widgets.HBox([
        widgets.VBox([labeled("Tone", w_tone)], layout=widgets.Layout(width="60%")),
        widgets.VBox([labeled("Word count", w_word_count)], layout=widgets.Layout(width="38%")),
    ], layout=widgets.Layout(justify_content="space-between")),
    labeled("Key points (one per line)", w_key_points),
    labeled("Call to action", w_cta),
    widgets.HBox([
        widgets.VBox([labeled("Hashtags", w_hashtags)], layout=widgets.Layout(width="49%")),
        widgets.VBox([labeled("Emojis", w_emojis)], layout=widgets.Layout(width="49%")),
    ], layout=widgets.Layout(justify_content="space-between")),
    widgets.HTML("<div style='height:8px'></div>"),
    widgets.HBox([w_generate_btn, widgets.HTML("<div style='width:14px'></div>"), w_status], layout=widgets.Layout(align_items="center")),
])
input_card.add_class("lp-card")

logs_card = widgets.VBox([
    widgets.HTML("<div class='lp-section-label'>Agent logs</div>"),
    agent_accordion,
])
logs_card.add_class("lp-card")

result_card = widgets.VBox([out_final])

display(widgets.VBox([input_card, logs_card, result_card]))